In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes datasets
!pip install optuna optuna-integration wandb

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-6y8h53sd/unsloth_1d4ea3c94ccd4b73bbd6378e4aeba05e
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-6y8h53sd/unsloth_1d4ea3c94ccd4b73bbd6378e4aeba05e
  Resolved https://github.com/unslothai/unsloth.git to commit 9a551831ef2eff33f947cc1a7e1b259f9913950b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
from google.colab import userdata
import optuna
from optuna.integration.wandb import WeightsAndBiasesCallback
import wandb
import torch
import gc
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
import os
os.environ["WANDB_PROJECT"] = "medical-llama3-hybrid-sweep"

In [ ]:
# 1. Authenticate with W&B Cloud securely via Colab Secrets
wandb_key = userdata.get('WANDB_API_KEY')
wandb.login(key=wandb_key)

# 2. Configure the W&B Optuna Callback
wandb_kwargs = {"project": "medical-llama3-hybrid-sweep"}
wandbc = WeightsAndBiasesCallback(
    wandb_kwargs=wandb_kwargs,
    metric_name="eval_loss",
    as_multirun=True
)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: akhilgalla (foobar41) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
/tmp/ipykernel_27808/900538074.py:7: ExperimentalWarning: WeightsAndBiasesCallback is experimental (supported from v2.9.0). The interface can change in the future.
  wandbc = WeightsAndBiasesCallback(


In [ ]:
def objective(trial):
    # 1. The ONLY Suggestion Block (Narrowed for Batch Size 32)
    lr = trial.suggest_float("learning_rate", 1e-4, 1e-3, log=True)
    rank = trial.suggest_categorical("lora_rank", [8, 16, 32])
    decay = trial.suggest_float("weight_decay", 0.05, 0.1)

    # 2. Seize control: Manually initialize W&B
    run = wandb.init(
        project="medical-llama3-hybrid-sweep-batch32-16bit",
        name=f"medical-sweep-trial-{trial.number}",
        reinit=True,
        config={
            "learning_rate": lr,
            "lora_rank": rank,
            "weight_decay": decay
        }
    )

    # Clear VRAM to prevent Out-Of-Memory crashes
    torch.cuda.empty_cache()
    gc.collect()

    # Load Fresh Base Model & Tokenizer
    max_seq_length = 2048
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="meta-llama/Meta-Llama-3-8B-Instruct",
        max_seq_length=max_seq_length,
        dtype=torch.bfloat16,
        load_in_4bit=False,
    )

    # Inject Optuna Rank
    model = FastLanguageModel.get_peft_model(
        model,
        r=rank,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        lora_alpha=32,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407
    )

    # Load the Micro-Dataset & Split
    dataset = load_dataset("medalpaca/medical_meadow_medical_flashcards", split="train[:1000]")

    llama3_prompt = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a highly accurate medical AI assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>
{}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
{}<|eot_id|>"""

    EOS_TOKEN = tokenizer.eos_token
    def format_flashcards(examples):
        inputs = examples["input"]
        outputs = examples["output"]
        texts = []
        for context, answer in zip(inputs, outputs):
            text = llama3_prompt.format(context, answer) + EOS_TOKEN
            texts.append(text)
        return { "text" : texts }

    dataset = dataset.map(format_flashcards, batched=True)
    split_dataset = dataset.train_test_split(test_size=0.2, seed=3407)
    train_data = split_dataset["train"]
    eval_data = split_dataset["test"]

    # A100 Max Saturation Config
    sft_config = SFTConfig(
        per_device_train_batch_size=32,
        gradient_accumulation_steps=2,
        num_train_epochs=1,
        eval_strategy="steps",
        eval_steps=1,          # Ensures 12 eval checkpoints across the 12 steps
        logging_steps=1,       # Forces W&B to plot every single step
        save_strategy="no",
        learning_rate=lr,
        weight_decay=decay,
        warmup_steps=2,        # Gets to the actual LR immediately
        fp16=False,
        bf16=True,
        optim="adamw_8bit",
        seed=3407,
        output_dir="/content/sweep_temp",
        report_to="wandb",
        # neftune_noise_alpha=5.0, # <--- DISABLED for pure sweep signal
        dataset_text_field="text",
        max_seq_length=max_seq_length,
        dataset_num_proc=2
    )

    trainer = SFTTrainer(
        model=model,
        processing_class=tokenizer,
        train_dataset=train_data,
        eval_dataset=eval_data,
        args=sft_config,
    )

    try:
        # 1. Execute the trial
        trainer.train()

        # 2. Extract the evaluation loss
        eval_loss = float("inf")
        for log in reversed(trainer.state.log_history):
            if "eval_loss" in log:
                eval_loss = log["eval_loss"]
                break

        wandb.finish()

        # 3. Clean up successfully trained models
        del model, tokenizer, trainer
        torch.cuda.empty_cache()
        gc.collect()

        return eval_loss

    except torch.OutOfMemoryError:
        # 4. AGGRESSIVE CLEANUP: Destroy the corrupted objects
        wandb.finish()

        # Delete the variables physically from Python's memory dictionary
        if 'model' in locals(): del model
        if 'tokenizer' in locals(): del tokenizer
        if 'trainer' in locals(): del trainer

        # Now the garbage collector can actually flush the meta tensors
        torch.cuda.empty_cache()
        gc.collect()

        print(f"\n⚠️ Trial {trial.number} OOM Crash. Corrupted tensors destroyed. Pruning...\n")
        raise optuna.exceptions.TrialPruned()


In [ ]:
# 4. Trigger the Warm-Started Optuna Study
pruner = optuna.pruners.HyperbandPruner()
study = optuna.create_study(direction="minimize", pruner=pruner)

# THE INJECTION: Force Optuna to test our scaled prediction on Trial 0
study.enqueue_trial({
    "learning_rate": 0.000590,
    "lora_rank": 16,
    "weight_decay": 0.075
})

[I 2026-03-07 04:34:43,648] A new study created in memory with name: no-name-f7065582-4f4e-4342-bd57-50960ead8ed3


In [ ]:
print("🚀 Starting Batch Size 32 Optuna Sweep...")
study.optimize(objective, n_trials=12)

print("\n🏆 Sweep Complete! Best Hyperparameters Found:")
print(study.best_trial.params)

🚀 Starting Batch Size 32 Optuna Sweep...


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


==((====))==  Unsloth 2026.3.3: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

...

Unsloth: Will load unsloth/llama-3-8b-Instruct as a legacy tokenizer.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 800 | Num Epochs = 1 | Total steps = 13
O^O/ \_/ \    Batch size per device = 32 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (32 x 2 x 1) = 64
 "-____-"     Trainable parameters = 20,971,520 of 8,051,232,768 (0.26% trained)


Step,Training Loss,Validation Loss
1,3.761238,3.664861
2,3.657425,2.173219
3,2.111188,1.852322
4,1.870835,2.209779
5,2.209476,1.413048
6,1.396611,1.071773
7,1.056749,1.072910
8,1.078501,1.021165
9,0.955435,0.916555
10,0.879861,0.883538


eval/loss,█▄▄▄▂▂▂▁▁▁▁▁▁
eval/runtime,█▆▇▇▅▂▂▄▆▇▁▅▅
eval/samples_per_second,▁▃▂▂▄▇▇▅▃▂█▄▄
eval/steps_per_second,▁▃▂▂▄▇▇▅▃▂█▄▄
train/epoch,▁▁▂▂▂▂▃▃▃▃▄▄▅▅▅▅▆▆▆▆▇▇█████
train/global_step,▁▁▂▂▂▂▃▃▃▃▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train/grad_norm,██▄▄▄▃▃▃▃▁▁▁▁
train/learning_rate,▁▅█▇▇▆▅▅▄▄▃▂▂
train/loss,██▄▃▄▂▁▂▁▁▁▁▁
eval/loss,0.83595
eval/runtime,5.0923


[I 2026-03-07 05:01:02,237] Trial 11 finished with value: 0.8359465003013611 and parameters: {'learning_rate': 0.0009445091811886635, 'lora_rank': 8, 'weight_decay': 0.08803359091479758}. Best is trial 9 with value: 0.8047744631767273.



🏆 Sweep Complete! Best Hyperparameters Found:
{'learning_rate': 0.0007099375152773817, 'lora_rank': 8, 'weight_decay': 0.07328643047092469}
